# PEFT: QLoRA for Text Generation

In this notebook, we show the differences between standard fine-tuning and PEFT using  **QLoRA (Quantized LoRA)** to fine-tune a **causal language model** for a **instruction tuning**.
We combine:
- 4-bit quantization (bitsandbytes)
- LoRA low-rank adapters (PEFT)

QLoRA setup will allow to fine-tune a multi-billion parameter LLM on a **single consumer GPU**.


## 1. Environment Setup

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.0 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

## 2. Load Dataset (Instruction Tuning)

We fine-tune a **causal language model** for **instruction-style generation**, using 
the [tatsu-lab/alpaca dataset](https://huggingface.co/datasets/tatsu-lab/alpaca/viewer/default/train?row=0) containing samples of instructions + responses. 

Each training sample consists of:
- A *prompt / instruction*
- A *target completion*


In [ ]:
dataset = load_dataset('tatsu-lab/alpaca', split='train[:2000]')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

## 3. Formatting Prompts

In [ ]:
def format_example(example):
    prompt = 'Instruction: ' + example['instruction'] + ' Input: ' + example['input'] + 'Output: '
    completion = example['output']
    return {'prompt': prompt, 'completion': completion}
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## 4. Load the Model

We will be using [facebook-opt-1.3b](https://huggingface.co/facebook/opt-1.3b), a 1.3B parameter model

#### 4.1 Load Full Model with no quantization for standard full fine-tuning

In [ ]:
model_name = 'facebook/opt-1.3b'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (v_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (out_proj): Linear(in_features=2048, out_features=2048, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192, out_features=2048, bias=True)
          (final_layer_norm): LayerN

#### 4.2 Load Quantized Base Model (4-bit) for QLoRA fine-tuning

we will be using [BitsAndBytes](https://huggingface.co/docs/transformers/en/main_classes/quantization#quantization) to quantize the weights of the original model into 4-bits

In [ ]:
model_name = 'facebook/opt-1.3b'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
            (v_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
            (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
            (out_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear4bit(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear4bit(in_features=8192, out_features=2048, bias=True)
          (f

## 5. Load the Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

## 6. Supervised Fine-Tuning (SFT)
We use **TRL's `SFTTrainer`**, which is optimized for instruction tuning and generation tasks.

#### 6.1 Standard Full SFT (No PEFT)


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    bf16=True,
    output_dir='./qlora-alpaca',
    save_strategy='epoch',
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

#### 6.2 PEFT SFT with QLoRA


##### LoRA configuration

We need to configure the parameters of LoRA (rank, to which modules LoRA is applied, ...), using the class `LoraConfig`. See https://huggingface.co/docs/peft/package_reference/lora for full documentation

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj','v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,420,288,000 || trainable%: 0.1107


##### Fine-tuning

After configuring LoRA, we fine-tune in the usual way

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    bf16=True,  
    output_dir='./qlora-alpaca',
    save_strategy='epoch',
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

## 7. Text Generation Test

In [ ]:
prompt = 'Instruction: ' + 'Explain LoRA in simple terms' + ' Input: ' + 'Output: '

### Response:'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Instruction: Explain LoRA in simple terms Input: 
Output: LoRA is a distributed, open-source, and collaborative research platform for the development of open-source software. It is a collaborative, open-source, and collaborative research platform for the development of open-source software. LoRA is a collaborative, open-source, and collaborative research platform for the development of open-source software.


## 8. Save the final model

In [ ]:
model.save_pretrained('sft_model')

## 12. Exercises (Advanced)
1. Increase model size (e.g. OPT‑2.7B or LLaMA‑style models)
2. Compare ranks r = 4, 8, 16
3. Measure GPU memory with and without quantization
4. Try merging LoRA weights and exporting the model
